In [3]:
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, TimestampType
from pyspark.sql.functions import window, sum as spark_sum, count, avg, col
import os
import shutil
import time
import pandas as pd

spark = SparkSession.builder \
    .appName("Pertemuan10-StructuredStreaming") \
    .master("local[*]") \
    .getOrCreate()
spark.sparkContext.setLogLevel("ERROR")

# Mengurangi jumlah partisi shuffle default (200) agar pemrosesan pada VM/laptop kita lebih cepat
# -- semakin kecil dataset (seperti simulasi kita), semakin tidak perlu partisi sebanyak itu
spark.conf.set("spark.sql.shuffle.partitions", "4")

print("SparkSession siap. Versi Spark:", spark.version)

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/11/12 13:43:43 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


SparkSession siap. Versi Spark: 3.5.9


In [4]:
# Menyiapkan folder kosong yang akan berperan sebagai "sumber data streaming"
folder_stream = "simulasi_stream_transaksi"
folder_checkpoint = "checkpoint_pertemuan10"

shutil.rmtree(folder_stream, ignore_errors=True)
shutil.rmtree(folder_checkpoint, ignore_errors=True)
os.makedirs(folder_stream, exist_ok=True)

print(f"Folder simulasi streaming '{folder_stream}' siap (kosong).")# Menyiapkan folder kosong yang akan berperan sebagai "sumber data streaming"
folder_stream = "simulasi_stream_transaksi"
folder_checkpoint = "checkpoint_pertemuan10"

shutil.rmtree(folder_stream, ignore_errors=True)
shutil.rmtree(folder_checkpoint, ignore_errors=True)
os.makedirs(folder_stream, exist_ok=True)

print(f"Folder simulasi streaming '{folder_stream}' siap (kosong).")

Folder simulasi streaming 'simulasi_stream_transaksi' siap (kosong).
Folder simulasi streaming 'simulasi_stream_transaksi' siap (kosong).


In [5]:
# Mendefinisikan skema secara eksplisit — wajib untuk streaming source berbasis berkas
skema_transaksi = StructType([
    StructField("trx_id", StringType(), True),
    StructField("kota", StringType(), True),
    StructField("kategori", StringType(), True),
    StructField("pendapatan", IntegerType(), True),
])

print("Skema berhasil didefinisikan:")
print(skema_transaksi)

Skema berhasil didefinisikan:
StructType([StructField('trx_id', StringType(), True), StructField('kota', StringType(), True), StructField('kategori', StringType(), True), StructField('pendapatan', IntegerType(), True)])


Membangun Streaming Query Pertama

In [6]:
# Mengidentifikasiinput source (readStream)

df_stream = spark.readStream \
    .format("csv") \
    .schema(skema_transaksi) \
    .option("header", True) \
    .load(folder_stream)

print("Streaming DataFrame berhasil didefinisikan (belum berjalan — baru 'rencana').")
print("df_stream.isStreaming:", df_stream.isStreaming)

Streaming DataFrame berhasil didefinisikan (belum berjalan — baru 'rencana').
df_stream.isStreaming: True


In [7]:
# Mendefinisikan transformasi

# Melakukan agregasi — sintaks IDENTIK dengan groupBy/agg pada DataFrame batch (Pertemuan 4-5)
agregasi_stream = df_stream.groupBy("kota").agg(
    spark_sum("pendapatan").alias("total_pendapatan"),
    count("trx_id").alias("jumlah_transaksi")
)

print("Transformasi berhasil didefinisikan.")

Transformasi berhasil didefinisikan.


In [8]:
# Mengidentifikasi output sink

query = agregasi_stream.writeStream \
    .outputMode("complete") \
    .format("memory") \
    .queryName("hasil_agregasi") \
    .start()

print("Streaming query dimulai! Status:", query.status)

Streaming query dimulai! Status: {'message': 'Getting offsets from FileStreamSource[file:/home/azka/praktikum-bigdata/pwertwmuan_10/praktikum10/simulasi_stream_transaksi]', 'isDataAvailable': False, 'isTriggerActive': True}


In [9]:
# Mengalirkan data baru dan mengamati hasilnya

# Batch data pertama "tiba"
pd.DataFrame({
    "trx_id": ["T1", "T2", "T3"],
    "kota": ["Magelang", "Solo", "Magelang"],
    "kategori": ["Elektronik", "Fashion", "Makanan"],
    "pendapatan": [150000, 200000, 75000],
}).to_csv(f"{folder_stream}/batch_1.csv", index=False)

print("Batch 1 dikirim. Menunggu Spark memproses...")
time.sleep(6)  # memberi waktu Spark memproses micro-batch (proses pertama kali butuh waktu ekstra)

spark.sql("SELECT * FROM hasil_agregasi ORDER BY total_pendapatan DESC").show()

Batch 1 dikirim. Menunggu Spark memproses...


[Stage 1:=============================>                             (2 + 2) / 4]

+----+----------------+----------------+
|kota|total_pendapatan|jumlah_transaksi|
+----+----------------+----------------+
+----+----------------+----------------+



In [10]:
# Batch data kedua "tiba" beberapa saat kemudian
pd.DataFrame({
    "trx_id": ["T4", "T5"],
    "kota": ["Magelang", "Yogyakarta"],
    "kategori": ["Fashion", "Elektronik"],
    "pendapatan": [300000, 500000],
}).to_csv(f"{folder_stream}/batch_2.csv", index=False)

print("Batch 2 dikirim. Menunggu Spark memproses...")
time.sleep(6)

spark.sql("SELECT * FROM hasil_agregasi ORDER BY total_pendapatan DESC").show()

Batch 2 dikirim. Menunggu Spark memproses...


+----------+----------------+----------------+
|      kota|total_pendapatan|jumlah_transaksi|
+----------+----------------+----------------+
|  Magelang|          525000|               3|
|Yogyakarta|          500000|               1|
|      Solo|          200000|               1|
+----------+----------------+----------------+



In [11]:
# Menghentikan streaming query

query.stop()
print("Streaming query dihentikan. Status aktif:", query.isActive)

Streaming query dihentikan. Status aktif: False


Agregasi Berbasis Window Waktu

In [12]:
skema_dengan_waktu = StructType([
    StructField("trx_id", StringType(), True),
    StructField("kota", StringType(), True),
    StructField("pendapatan", IntegerType(), True),
    StructField("waktu_transaksi", TimestampType(), True),
])

# Membuat ulang folder simulasi (kosong) untuk demo window waktu
folder_stream2 = "simulasi_stream_waktu"
shutil.rmtree(folder_stream2, ignore_errors=True)
os.makedirs(folder_stream2, exist_ok=True)

df_stream2 = spark.readStream \
    .format("csv") \
    .schema(skema_dengan_waktu) \
    .option("header", True) \
    .load(folder_stream2)

# Mengelompokkan berdasarkan window waktu 1 menit, dihitung dari kolom waktu_transaksi
agregasi_window = df_stream2.groupBy(
    window(col("waktu_transaksi"), "1 minute")
).agg(
    spark_sum("pendapatan").alias("total_pendapatan"),
    count("trx_id").alias("jumlah_transaksi")
)

query2 = agregasi_window.writeStream \
    .outputMode("complete") \
    .format("memory") \
    .queryName("hasil_window") \
    .start()

print("Streaming query window waktu dimulai.")

Streaming query window waktu dimulai.


In [13]:
import pandas as pd
import time
from datetime import datetime

In [14]:
from datetime import datetime

# Mengirim data dengan timestamp saat ini
waktu_sekarang = datetime.now()
pd.DataFrame({
    "trx_id": ["W1", "W2"],
    "kota": ["Semarang", "Solo"],
    "pendapatan": [120000, 180000],
    "waktu_transaksi": [waktu_sekarang.strftime("%Y-%m-%d %H:%M:%S")] * 2,
}).to_csv(f"{folder_stream2}/batch_w1.csv", index=False)

time.sleep(6)
spark.sql("SELECT window.start, window.end, total_pendapatan, jumlah_transaksi FROM hasil_window").show(truncate=False)

+-------------------+-------------------+----------------+----------------+
|start              |end                |total_pendapatan|jumlah_transaksi|
+-------------------+-------------------+----------------+----------------+
|2026-11-12 13:52:00|2026-11-12 13:53:00|300000          |2               |
+-------------------+-------------------+----------------+----------------+

